In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'
print(sorted(os.listdir(BASE_PATH)))

def load(keyword):
    files = [f for f in os.listdir(BASE_PATH) if keyword in f.lower()]
    print(keyword, '->', files)
    return pd.read_csv(BASE_PATH + files[0])

branches = load('branch')
products = load('product')
print('branches', branches.shape)
print(list(branches.columns))
print('products', products.shape)
print(list(products.columns))
print(products.head(2).T)

Mounted at /content/drive
['branches.csv', 'customers.csv', 'inventory_master.csv', 'invoices.csv', 'payments.csv', 'products.csv', 'purchase_orders_header.csv', 'purchase_orders_lines.csv', 'sales_orders_header.csv', 'sales_orders_lines.csv', 'stock_ledger.csv', 'suppliers.csv']
branch -> ['branches.csv']
product -> ['products.csv']
branches (6, 13)
['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']
products (30, 23)
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']
                   

In [2]:
# Cell 2: check dimensions_cm, weight_kg and uom for all 30 products
parts = products['dimensions_cm'].astype(str).str.lower().str.split('x', expand=True)
print("Products:", len(products))
print("Rows that split into exactly 3 parts:", int(parts.notna().all(axis=1).sum()) if parts.shape[1] == 3 else 0)

dims = parts.apply(pd.to_numeric, errors='coerce')
products['length_cm'] = dims[0]
products['width_cm'] = dims[1]
products['height_cm'] = dims[2]
products['unit_volume_cm3'] = dims[0] * dims[1] * dims[2]

print("Empty dimensions_cm:", int(products['dimensions_cm'].isnull().sum()))
print("Dimension values that failed to convert:", int(dims.isnull().sum().sum()))
print("Dimension values zero or below:", int((dims <= 0).sum().sum()))
print("Empty weight_kg:", int(products['weight_kg'].isnull().sum()))
print("uom values:", products['uom'].value_counts().to_dict())
print("Smallest unit volume (cm3):", products['unit_volume_cm3'].min())
print("Largest unit volume (cm3):", products['unit_volume_cm3'].max())

Products: 30
Rows that split into exactly 3 parts: 30
Empty dimensions_cm: 0
Dimension values that failed to convert: 0
Dimension values zero or below: 0
Empty weight_kg: 0
uom values: {'piece': 24, 'set': 5, 'cartridge': 1}
Smallest unit volume (cm3): 256
Largest unit volume (cm3): 728000


In [3]:
# Cell 3: the 6 products not counted in pieces, plus smallest and largest volume
cols = ['product_id', 'product_name', 'uom', 'dimensions_cm', 'unit_volume_cm3', 'weight_kg', 'max_stock_level']

print("Products not counted in pieces:")
print(products.loc[products['uom'] != 'piece', cols].to_string(index=False))

print("\nFive smallest unit volumes:")
print(products.sort_values('unit_volume_cm3').head(5)[cols].to_string(index=False))

print("\nFive largest unit volumes:")
print(products.sort_values('unit_volume_cm3', ascending=False).head(5)[cols].to_string(index=False))

Products not counted in pieces:
product_id                 product_name       uom dimensions_cm  unit_volume_cm3  weight_kg  max_stock_level
      P004            Track Chain TC-45       set     180x40x35           252000     210.00               20
      P012              Brake Pad BP-40       set      20x12x10             2400       3.20              120
      P013               Seal Kit SK-15       set       15x10x6              900       0.60              200
      P014 Transmission Assembly TA-800       set     190x60x55           627000     260.00               12
      P026        Grease Cartridge GC-5 cartridge        25x6x6              900       0.45              500
      P027             O-Ring Set OR-50       set         8x8x4              256       0.20              400

Five smallest unit volumes:
product_id            product_name   uom dimensions_cm  unit_volume_cm3  weight_kg  max_stock_level
      P027        O-Ring Set OR-50   set         8x8x4              256     

In [4]:
# Cell 4: turn warehouse_capacity text into numbers
branches['capacity_sqft'] = pd.to_numeric(
    branches['warehouse_capacity'].astype(str).str.lower().str.replace('sqft', '', regex=False).str.strip(),
    errors='coerce')

print(branches[['branch_id', 'warehouse_type', 'warehouse_capacity', 'capacity_sqft']].to_string(index=False))
print("\nBranches that failed to convert:", int(branches['capacity_sqft'].isnull().sum()))
print("Capacity zero or below:", int((branches['capacity_sqft'] <= 0).sum()))
print("Total capacity (sqft):", branches['capacity_sqft'].sum())

branch_id warehouse_type warehouse_capacity  capacity_sqft
   DEL001        Central         45230 sqft          45230
   PUN001       Regional         38750 sqft          38750
   CHN001       Regional         41890 sqft          41890
   HYD001         Branch         36210 sqft          36210
   KOL001         Branch         29880 sqft          29880
   AHM001    Service Hub         34120 sqft          34120

Branches that failed to convert: 0
Capacity zero or below: 0
Total capacity (sqft): 226080


In [5]:
# Cell 5: load the four order tables needed for stock per branch
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')

print("so_h (sales order headers):", so_h.shape)
print("so_l (sales order lines):", so_l.shape)
print("po_h (purchase order headers):", po_h.shape)
print("po_l (purchase order lines):", po_l.shape)

so_h (sales order headers): (20000, 11)
so_l (sales order lines): (130402, 9)
po_h (purchase order headers): (24000, 10)
po_l (purchase order lines): (155495, 9)


In [6]:
# Cell 6: load the stock ledger and the inventory master
ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')

print("ledger (stock ledger):", ledger.shape)
print("inv (inventory master):", inv.shape)

ledger (stock ledger): (237230, 9)
inv (inventory master): (180, 8)


In [7]:
# look at the inventory master columns and the ledger movement types before building stock per branch
print("inv columns:", list(inv.columns))
print()
print(inv.head(3).to_string())
print()
print("inv rows:", len(inv), "| products:", inv['product_id'].nunique(), "| branches:", inv['branch_id'].nunique())
print()
print("ledger movement_type counts:")
print(ledger['movement_type'].value_counts().to_string())
print()
print("ledger reference_type counts:")
print(ledger['reference_type'].value_counts().to_string())

inv columns: ['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

  product_id branch_id  opening_stock  reorder_level  safety_stock  max_stock  current_stock warehouse_bin
0       P001    DEL001            196             59            46        340         115928           C15
1       P001    PUN001            198             70            43        349         109389           D16
2       P001    CHN001            187             62            44        311          99373           A11

inv rows: 180 | products: 30 | branches: 6

ledger movement_type counts:
movement_type
IN            127611
OUT           107165
ADJUSTMENT      2454

ledger reference_type counts:
reference_type
PO     127611
SO     107165
ADJ      2454


In [8]:
# stock per product and branch at the end of the data: opening + received - delivered + net adjustments
so = so_l.merge(so_h[['so_id', 'branch_id', 'order_status']], on='so_id', how='left')
so = so[so['order_status'] == 'Delivered']
units_out = so.groupby(['product_id', 'branch_id'])['quantity'].sum().rename('units_out')

po = po_l.merge(po_h[['po_id', 'branch_id', 'po_status']], on='po_id', how='left')
po = po[po['po_status'] == 'Received']
units_in = po.groupby(['product_id', 'branch_id'])['quantity'].sum().rename('units_in')

led_in = ledger[ledger['movement_type'] == 'IN'].groupby(['product_id', 'branch_id'])['quantity'].sum().rename('ledger_in')
led_out = ledger[ledger['movement_type'] == 'OUT'].groupby(['product_id', 'branch_id'])['quantity'].sum().rename('ledger_out')
last_bal = ledger.groupby(['product_id', 'branch_id'])['running_balance'].last().rename('last_balance')

pair = inv.set_index(['product_id', 'branch_id'])[['opening_stock', 'current_stock']]
pair = pair.join([units_in, units_out, led_in, led_out, last_bal])
print("pairs with no ledger balance:", int(pair['last_balance'].isna().sum()))
pair[['units_in', 'units_out', 'ledger_in', 'ledger_out']] = pair[['units_in', 'units_out', 'ledger_in', 'ledger_out']].fillna(0)

pair['net_adj'] = pair['last_balance'] - pair['opening_stock'] - pair['ledger_in'] + pair['ledger_out']
pair['stock_end'] = pair['opening_stock'] + pair['units_in'] - pair['units_out'] + pair['net_adj']

print("pairs:", len(pair))
print("opening stock total:", int(pair['opening_stock'].sum()))
print("units received total:", int(pair['units_in'].sum()))
print("units delivered total:", int(pair['units_out'].sum()))
print("net adjustments total:", int(pair['net_adj'].sum()))
print("stock at the end total:", int(pair['stock_end'].sum()))
print("current_stock column total (for comparison only):", int(pair['current_stock'].sum()))
print()
print(pair.groupby('branch_id')['stock_end'].sum().astype(int).to_string())

pairs with no ledger balance: 0
pairs: 180
opening stock total: 32969
units received total: 22387123
units delivered total: 1234939
net adjustments total: 62
stock at the end total: 21185215
current_stock column total (for comparison only): 19303266

branch_id
AHM001    3472113
CHN001    3548773
DEL001    3597678
HYD001    3555196
KOL001    3435814
PUN001    3575641


In [9]:
# units per sqft per branch at the end of the data: stock divided by warehouse size
dens = pair.groupby('branch_id')['stock_end'].sum().rename('stock_units').to_frame()
dens = dens.join(branches.set_index('branch_id')[['warehouse_type', 'capacity_sqft']])
dens['units_per_sqft'] = (dens['stock_units'] / dens['capacity_sqft']).round(1)
dens = dens.sort_values('units_per_sqft', ascending=False)
print(dens.to_string())
print()
print("most crowded vs least crowded:", round(dens['units_per_sqft'].max() / dens['units_per_sqft'].min(), 2), "times")

           stock_units warehouse_type  capacity_sqft  units_per_sqft
branch_id                                                           
KOL001         3435814         Branch          29880           115.0
AHM001         3472113    Service Hub          34120           101.8
HYD001         3555196         Branch          36210            98.2
PUN001         3575641       Regional          38750            92.3
CHN001         3548773       Regional          41890            84.7
DEL001         3597678        Central          45230            79.5

most crowded vs least crowded: 1.45 times


In [10]:
# look at the ledger adjustment rows: sign, total and total per year
adj = ledger[ledger['movement_type'] == 'ADJUSTMENT'].copy()
adj['year'] = pd.to_datetime(adj['movement_date']).dt.year
print("adjustment rows:", len(adj))
print("smallest quantity:", adj['quantity'].min(), "| largest quantity:", adj['quantity'].max())
print("negative:", int((adj['quantity'] < 0).sum()), "| zero:", int((adj['quantity'] == 0).sum()), "| positive:", int((adj['quantity'] > 0).sum()))
print("total of all adjustment quantities:", int(adj['quantity'].sum()))
print()
print("total per year:")
print(adj.groupby('year')['quantity'].agg(['count', 'sum']).to_string())

adjustment rows: 2454
smallest quantity: 5 | largest quantity: 50
negative: 0 | zero: 0 | positive: 2454
total of all adjustment quantities: 66990

total per year:
      count    sum
year              
2019    384  10590
2020    391  10509
2021    420  11598
2022    440  11940
2023    421  11744
2024    398  10609


In [11]:
# find the real direction of each ledger row from the change in running balance
led = ledger[['product_id', 'branch_id', 'movement_type', 'quantity', 'running_balance']].copy()
led['prev_balance'] = led.groupby(['product_id', 'branch_id'])['running_balance'].shift()
first = led['prev_balance'].isna()
open_map = inv.set_index(['product_id', 'branch_id'])['opening_stock']
keys = pd.MultiIndex.from_frame(led.loc[first, ['product_id', 'branch_id']])
led.loc[first, 'prev_balance'] = open_map.reindex(keys).values
led['change'] = led['running_balance'] - led['prev_balance']

led['what_happened'] = 'something else'
led.loc[led['change'] == led['quantity'], 'what_happened'] = 'balance went up by quantity'
led.loc[led['change'] == -led['quantity'], 'what_happened'] = 'balance went down by quantity'

print(led.groupby(['movement_type', 'what_happened']).size().unstack(fill_value=0).to_string())
print()
adj_rows = led[led['movement_type'] == 'ADJUSTMENT']
print("adjustment rows that added stock:", int((adj_rows['change'] > 0).sum()))
print("adjustment rows that removed stock:", int((adj_rows['change'] < 0).sum()))
print("signed total of adjustment changes:", int(adj_rows['change'].sum()))

what_happened  balance went down by quantity  balance went up by quantity
movement_type                                                            
ADJUSTMENT                              1227                         1227
IN                                         0                       127611
OUT                                   107165                            0

adjustment rows that added stock: 1227
adjustment rows that removed stock: 1227
signed total of adjustment changes: 62


In [12]:
# stock per branch at each year end 2019 to 2024: opening + received - delivered + signed adjustments, added up year by year
sd = so_l.merge(so_h[['so_id', 'branch_id', 'order_status', 'delivery_date']], on='so_id', how='left')
sd = sd[sd['order_status'] == 'Delivered'].copy()
sd['year'] = pd.to_datetime(sd['delivery_date']).dt.year

pr = po_l.merge(po_h[['po_id', 'branch_id', 'po_status', 'received_date']], on='po_id', how='left')
pr = pr[pr['po_status'] == 'Received'].copy()
pr['year'] = pd.to_datetime(pr['received_date']).dt.year

ad = led[led['movement_type'] == 'ADJUSTMENT'].copy()
ad['year'] = pd.to_datetime(ledger.loc[ad.index, 'movement_date']).dt.year

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
in_y = pr.groupby(['branch_id', 'year'])['quantity'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0)
out_y = sd.groupby(['branch_id', 'year'])['quantity'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0)
adj_y = ad.groupby(['branch_id', 'year'])['change'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0)
open_b = inv.groupby('branch_id')['opening_stock'].sum()

stock_ye = (in_y - out_y + adj_y).cumsum(axis=1).add(open_b, axis=0).astype(int)
print("stock at the end of each year per branch (2025 = end of data):")
print(stock_ye.to_string())
print()
expected = [3384660, 6960247, 10529943, 14009405, 17557387, 20993633]
for y, e in zip(years[:6], expected):
    print(y, "total:", int(stock_ye[y].sum()), "| expected:", e, "| difference:", int(stock_ye[y].sum()) - e)
print()
end_diff = stock_ye[2025] - pair.groupby('branch_id')['stock_end'].sum()
print("largest difference between 2025 end and stock at the end per branch:", int(end_diff.abs().max()))

stock at the end of each year per branch (2025 = end of data):
year         2019     2020     2021     2022     2023     2024     2025
branch_id                                                              
AHM001     552716  1142747  1764206  2297090  2858247  3442316  3472113
CHN001     564933  1200597  1727213  2322200  2914591  3516571  3548773
DEL001     593228  1151252  1732555  2362474  2971460  3566310  3597678
HYD001     549018  1141894  1798295  2367793  2968447  3517178  3555196
KOL001     561067  1134619  1682062  2281961  2868931  3400815  3435814
PUN001     563698  1189138  1825612  2377887  2975711  3550443  3575641

2019 total: 3384660 | expected: 3384660 | difference: 0
2020 total: 6960247 | expected: 6960247 | difference: 0
2021 total: 10529943 | expected: 10529943 | difference: 0
2022 total: 14009405 | expected: 14009405 | difference: 0
2023 total: 17557387 | expected: 17557387 | difference: 0
2024 total: 20993633 | expected: 20993633 | difference: 0

largest differe

In [13]:
# units per sqft per branch at each year end: stock divided by warehouse size
size = branches.set_index('branch_id')['capacity_sqft']
dens_y = stock_ye.div(size, axis=0).round(1)
print("units per sqft at each year end (2025 = end of data):")
print(dens_y.to_string())
print()
print("branches from most crowded to least crowded each year:")
for y in dens_y.columns:
    print(y, list(dens_y[y].sort_values(ascending=False).index))
print()
print("most crowded vs least crowded each year:")
print((dens_y.max() / dens_y.min()).round(2).to_string())

units per sqft at each year end (2025 = end of data):
year       2019  2020  2021  2022  2023   2024   2025
branch_id                                            
AHM001     16.2  33.5  51.7  67.3  83.8  100.9  101.8
CHN001     13.5  28.7  41.2  55.4  69.6   83.9   84.7
DEL001     13.1  25.5  38.3  52.2  65.7   78.8   79.5
HYD001     15.2  31.5  49.7  65.4  82.0   97.1   98.2
KOL001     18.8  38.0  56.3  76.4  96.0  113.8  115.0
PUN001     14.5  30.7  47.1  61.4  76.8   91.6   92.3

branches from most crowded to least crowded each year:
2019 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2020 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2021 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2022 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2023 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2024 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001']
2025 ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL00

In [14]:
# stock volume per branch at the end of the data and average pile height if spread over the floor
vol = pair.reset_index().merge(products[['product_id', 'unit_volume_cm3']], on='product_id', how='left')
print("pairs with no unit volume:", int(vol['unit_volume_cm3'].isna().sum()))
vol['volume_m3'] = vol['stock_end'] * vol['unit_volume_cm3'] / 1000000

vb = vol.groupby('branch_id')['volume_m3'].sum().rename('stock_volume_m3').to_frame()
vb = vb.join(branches.set_index('branch_id')[['capacity_sqft']])
vb['floor_area_m2'] = vb['capacity_sqft'] * 0.092903
vb['pile_height_m'] = vb['stock_volume_m3'] / vb['floor_area_m2']
vb = vb.round(1).sort_values('pile_height_m', ascending=False)
print(vb.to_string())
print()
print("most piled vs least piled:", round(vb['pile_height_m'].max() / vb['pile_height_m'].min(), 2), "times")

pairs with no unit volume: 0
           stock_volume_m3  capacity_sqft  floor_area_m2  pile_height_m
branch_id                                                              
KOL001            227728.2          29880         2775.9           82.0
AHM001            233904.3          34120         3169.9           73.8
HYD001            242707.2          36210         3364.0           72.1
PUN001            239881.1          38750         3600.0           66.6
CHN001            247324.9          41890         3891.7           63.6
DEL001            237823.5          45230         4202.0           56.6

most piled vs least piled: 1.45 times


In [15]:
# which products make up the stock volume at the end of the data (all branches together)
pv = vol.groupby('product_id').agg(units=('stock_end', 'sum'), unit_volume_cm3=('unit_volume_cm3', 'first'), volume_m3=('volume_m3', 'sum'))
pv = pv.join(products.set_index('product_id')[['product_name', 'uom']])
pv['share_of_volume_pct'] = (pv['volume_m3'] / pv['volume_m3'].sum() * 100).round(1)
pv['volume_m3'] = pv['volume_m3'].round(1)
pv = pv.sort_values('volume_m3', ascending=False)
print(pv[['product_name', 'uom', 'units', 'unit_volume_cm3', 'volume_m3', 'share_of_volume_pct']].head(8).to_string())
print()
print("top 2 products share of volume %:", round(pv['share_of_volume_pct'].head(2).sum(), 1))
print("top 5 products share of volume %:", round(pv['share_of_volume_pct'].head(5).sum(), 1))
print("total volume m3:", round(pv['volume_m3'].sum(), 1))

                            product_name    uom   units  unit_volume_cm3  volume_m3  share_of_volume_pct
product_id                                                                                              
P015                 Engine Block EB-600  piece  694538           728000   505623.7                 35.4
P014        Transmission Assembly TA-800    set  724530           627000   454280.3                 31.8
P004                   Track Chain TC-45    set  700277           252000   176469.8                 12.3
P016                     Radiator RD-250  piece  693262            72000    49914.9                  3.5
P018                     Wheel Rim WR-24  piece  709520            43750    31041.5                  2.2
P021                Boom Cylinder BC-400  piece  703315            42120    29623.6                  2.1
P011           Hydraulic Cylinder HC-200  piece  713667            38880    27747.4                  1.9
P023                   Idler Wheel IW-55  piece  718664

In [16]:
# pile height per branch with all products, without the top 2 and without the top 5 by volume
area = size * 0.092903
top2 = ['P015', 'P014']
top5 = ['P015', 'P014', 'P004', 'P016', 'P018']
sens = pd.DataFrame({
    'all products': vol.groupby('branch_id')['volume_m3'].sum() / area,
    'without top 2': vol[~vol['product_id'].isin(top2)].groupby('branch_id')['volume_m3'].sum() / area,
    'without top 5': vol[~vol['product_id'].isin(top5)].groupby('branch_id')['volume_m3'].sum() / area,
}).round(1)
print("average pile height in metres:")
print(sens.to_string())
print()
for col in sens.columns:
    print(col, "| order:", list(sens[col].sort_values(ascending=False).index), "| most vs least:", round(sens[col].max() / sens[col].min(), 2))

average pile height in metres:
           all products  without top 2  without top 5
branch_id                                            
AHM001             73.8           24.2           11.1
CHN001             63.6           20.1            8.9
DEL001             56.6           19.2            8.7
HYD001             72.1           23.3           10.3
KOL001             82.0           27.7           12.4
PUN001             66.6           21.8           10.1

all products | order: ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001'] | most vs least: 1.45
without top 2 | order: ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001'] | most vs least: 1.44
without top 5 | order: ['KOL001', 'AHM001', 'HYD001', 'PUN001', 'CHN001', 'DEL001'] | most vs least: 1.43


In [17]:
# check 1: empty, zero or negative values and date conversion failures in the tables behind stock per branch
print("inventory master rows:", len(inv))
print("opening stock empty:", int(inv['opening_stock'].isna().sum()), "| zero or below 0:", int((inv['opening_stock'] <= 0).sum()))
print("product or branch empty:", int(inv['product_id'].isna().sum() + inv['branch_id'].isna().sum()))
print()
print("ledger rows:", len(ledger))
print("quantity empty:", int(ledger['quantity'].isna().sum()), "| zero or below 0:", int((ledger['quantity'] <= 0).sum()))
print("running balance empty:", int(ledger['running_balance'].isna().sum()), "| below 0:", int((ledger['running_balance'] < 0).sum()))
print("ledger dates failing to convert:", int(pd.to_datetime(ledger['movement_date'], errors='coerce').isna().sum()))
print()
print("delivered lines:", len(sd), "| delivery dates failing to convert:", int(sd['year'].isna().sum()))
print("received lines:", len(pr), "| received dates failing to convert:", int(pr['year'].isna().sum()))

inventory master rows: 180
opening stock empty: 0 | zero or below 0: 0
product or branch empty: 0

ledger rows: 237230
quantity empty: 0 | zero or below 0: 0
running balance empty: 0 | below 0: 0
ledger dates failing to convert: 0

delivered lines: 117656 | delivery dates failing to convert: 0
received lines: 140077 | received dates failing to convert: 0


In [18]:
# check 2: stock per product and branch at every year end must never be zero or below 0, and must add up to the branch table
keys = ['product_id', 'branch_id']
in_p = pr.groupby(keys + ['year'])['quantity'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0).reindex(pair.index, fill_value=0)
out_p = sd.groupby(keys + ['year'])['quantity'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0).reindex(pair.index, fill_value=0)
adj_p = ad.groupby(keys + ['year'])['change'].sum().unstack(fill_value=0).reindex(columns=years, fill_value=0).reindex(pair.index, fill_value=0)
stock_p = (in_p - out_p + adj_p).cumsum(axis=1).add(pair['opening_stock'], axis=0)

print("duplicate product and branch pairs in the inventory master:", int(inv.duplicated(keys).sum()))
print("pairs:", len(stock_p), "| year ends:", stock_p.shape[1], "| values checked:", stock_p.size)
print("values below 0:", int((stock_p < 0).sum().sum()), "| values at 0:", int((stock_p == 0).sum().sum()))
low = stock_p.stack()
print("smallest stock value:", int(low.min()), "| at (product, branch, year):", low.idxmin())
print("largest difference between pair sums and the branch table:", int((stock_p.groupby('branch_id').sum() - stock_ye).abs().max().max()))

duplicate product and branch pairs in the inventory master: 0
pairs: 180 | year ends: 7 | values checked: 1260
values below 0: 0 | values at 0: 0
smallest stock value: 13616 | at (product, branch, year): ('P029', 'HYD001', np.int64(2019))
largest difference between pair sums and the branch table: 0


In [19]:
# check 3: repeated ids and identical copies of whole rows in the tables behind stock
print("ledger movement ids repeated:", int(ledger['movement_id'].duplicated().sum()))
print("sales order ids repeated in the header table:", int(so_h['so_id'].duplicated().sum()), "of", len(so_h))
print("purchase order ids repeated in the header table:", int(po_h['po_id'].duplicated().sum()), "of", len(po_h))
print("identical whole rows in sales order lines:", int(so_l.duplicated().sum()), "of", len(so_l))
print("identical whole rows in purchase order lines:", int(po_l.duplicated().sum()), "of", len(po_l))
print()
print("sales order line columns:", list(so_l.columns))
print("purchase order line columns:", list(po_l.columns))

ledger movement ids repeated: 0
sales order ids repeated in the header table: 0 of 20000
purchase order ids repeated in the header table: 0 of 24000
identical whole rows in sales order lines: 0 of 130402
identical whole rows in purchase order lines: 0 of 155495

sales order line columns: ['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']
purchase order line columns: ['po_id', 'line_number', 'product_id', 'quantity', 'unit_cost', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']


In [20]:
# check 3 (second test): how many units sit in exact copies of a line, and how big that is against stock
sk = ['so_id', 'product_id', 'quantity', 'unit_price']
gs = sd.groupby(sk).size()
extra_sales_units = int(((gs - 1) * gs.index.get_level_values('quantity')).sum())
print("sales: groups with an exact copy:", int((gs > 1).sum()), "| extra lines:", int((gs - 1).sum()))
print("sales: extra units in the copies:", extra_sales_units)
print("sales: extra units as % of delivered units:", round(extra_sales_units / sd['quantity'].sum() * 100, 2))
print("sales: extra units as % of stock at the end:", round(extra_sales_units / pair['stock_end'].sum() * 100, 3))
print()
pk = ['po_id', 'product_id', 'quantity', 'unit_cost']
gp = pr.groupby(pk).size()
print("purchases: groups with an exact copy:", int((gp > 1).sum()))

sales: groups with an exact copy: 686 | extra lines: 688
sales: extra units in the copies: 7604
sales: extra units as % of delivered units: 0.62
sales: extra units as % of stock at the end: 0.036

purchases: groups with an exact copy: 0


In [21]:
# check 3 (third test): reconcile 686 groups with the 684 pairs found in KPI 11
cg = gs[gs > 1].reset_index(name='lines')
pairs_with_copy = cg.groupby(['so_id', 'product_id']).size()
print("groups with an exact copy:", len(cg), "| extra lines:", int((cg['lines'] - 1).sum()))
print("order and product pairs with an exact copy:", len(pairs_with_copy))
print("pairs holding 2 or more different copy groups:", int((pairs_with_copy > 1).sum()))
print("groups made of 3 or more identical lines:", int((cg['lines'] > 2).sum()))

groups with an exact copy: 686 | extra lines: 688
order and product pairs with an exact copy: 684
pairs holding 2 or more different copy groups: 2
groups made of 3 or more identical lines: 2


In [22]:
# check 4: totals reconcile from raw lines to the final stock, volume and warehouse size
st = so_l.merge(so_h[['so_id', 'order_status']], on='so_id', how='left')
pt = po_l.merge(po_h[['po_id', 'po_status']], on='po_id', how='left')
print("sales order lines by status (lines, units):")
print(st.groupby('order_status')['quantity'].agg(['count', 'sum']).to_string())
print("purchase order lines by status (lines, units):")
print(pt.groupby('po_status')['quantity'].agg(['count', 'sum']).to_string())
print()
left_s = st['order_status'] != 'Delivered'
left_p = pt['po_status'] != 'Received'
print("sales lines: all", len(st), "= counted", len(sd), "+ left out", int(left_s.sum()), "| difference", len(st) - len(sd) - int(left_s.sum()))
print("sales units: all", int(st['quantity'].sum()), "= counted", int(sd['quantity'].sum()), "+ left out", int(st.loc[left_s, 'quantity'].sum()), "| difference", int(st['quantity'].sum() - sd['quantity'].sum() - st.loc[left_s, 'quantity'].sum()))
print("purchase lines: all", len(pt), "= counted", len(pr), "+ left out", int(left_p.sum()), "| difference", len(pt) - len(pr) - int(left_p.sum()))
print("purchase units: all", int(pt['quantity'].sum()), "= counted", int(pr['quantity'].sum()), "+ left out", int(pt.loc[left_p, 'quantity'].sum()), "| difference", int(pt['quantity'].sum() - pr['quantity'].sum() - pt.loc[left_p, 'quantity'].sum()))
print()
long_way = int(inv['opening_stock'].sum() + pr['quantity'].sum() - sd['quantity'].sum() + ad['change'].sum())
print("stock the long way (opening + received - delivered + adjustments):", long_way)
print("stock in pair table:", int(pair['stock_end'].sum()), "| year-end table 2025:", int(stock_ye[2025].sum()), "| units per sqft table:", int(dens['stock_units'].sum()))
print()
prod_units = inv.groupby('product_id')['opening_stock'].sum() + pr.groupby('product_id')['quantity'].sum() - sd.groupby('product_id')['quantity'].sum() + ad.groupby('product_id')['change'].sum()
vol_alt = (prod_units * products.set_index('product_id')['unit_volume_cm3']).sum() / 1000000
print("volume m3 from the pair table:", round(vol['volume_m3'].sum(), 1), "| second route from units per product:", round(vol_alt, 1))
print("warehouse size total sqft: branches table", int(branches['capacity_sqft'].sum()), "| units per sqft table", int(dens['capacity_sqft'].sum()))

sales order lines by status (lines, units):
               count      sum
order_status                 
Cancelled      12746   133595
Delivered     117656  1234939
purchase order lines by status (lines, units):
            count       sum
po_status                  
Cancelled   15418   2480593
Received   140077  22387123

sales lines: all 130402 = counted 117656 + left out 12746 | difference 0
sales units: all 1368534 = counted 1234939 + left out 133595 | difference 0
purchase lines: all 155495 = counted 140077 + left out 15418 | difference 0
purchase units: all 24867716 = counted 22387123 + left out 2480593 | difference 0

stock the long way (opening + received - delivered + adjustments): 21185215
stock in pair table: 21185215 | year-end table 2025: 21185215 | units per sqft table: 21185215

volume m3 from the pair table: 1429369.3 | second route from units per product: 1429369.3
warehouse size total sqft: branches table 226080 | units per sqft table 226080


In [23]:
# units per sqft at 2019, 2024 and the end of the data, rebuilt a second way
so_branch = dict(zip(so_h['so_id'], so_h['branch_id']))
po_branch = dict(zip(po_h['po_id'], po_h['branch_id']))
del_ids = set(so_h.loc[so_h['order_status'] == 'Delivered', 'so_id'])
rec_ids = set(po_h.loc[po_h['po_status'] == 'Received', 'po_id'])
del_year = dict(zip(so_h['so_id'], pd.to_datetime(so_h['delivery_date']).dt.year))
rec_year = dict(zip(po_h['po_id'], pd.to_datetime(po_h['received_date']).dt.year))

a = so_l[so_l['so_id'].isin(del_ids)]
b = po_l[po_l['po_id'].isin(rec_ids)]
sa = pd.DataFrame({'branch': a['so_id'].map(so_branch), 'year': a['so_id'].map(del_year), 'units': a['quantity']})
pb = pd.DataFrame({'branch': b['po_id'].map(po_branch), 'year': b['po_id'].map(rec_year), 'units': b['quantity']})
is_adj = led['movement_type'] == 'ADJUSTMENT'
aj = pd.DataFrame({'branch': led.loc[is_adj, 'branch_id'], 'year': pd.to_datetime(ledger.loc[is_adj, 'movement_date']).dt.year, 'units': led.loc[is_adj, 'change']})

open_b2 = inv.groupby('branch_id')['opening_stock'].sum()
cap2 = branches.set_index('branch_id')['warehouse_capacity'].str.extract(r'(\d+)')[0].astype(int)

res2 = {}
for y in [2019, 2024, 2025]:
    s_in = pb[pb['year'] <= y].groupby('branch')['units'].sum()
    s_out = sa[sa['year'] <= y].groupby('branch')['units'].sum()
    s_adj = aj[aj['year'] <= y].groupby('branch')['units'].sum()
    res2[y] = ((open_b2 + s_in - s_out + s_adj) / cap2).round(1)
second = pd.DataFrame(res2)
print("units per sqft, second method (2025 = end of data):")
print(second.to_string())
print()
print("largest difference vs the table dens_y:", float((second - dens_y[[2019, 2024, 2025]]).abs().max().max()))

units per sqft, second method (2025 = end of data):
           2019   2024   2025
branch_id                    
AHM001     16.2  100.9  101.8
CHN001     13.5   83.9   84.7
DEL001     13.1   78.8   79.5
HYD001     15.2   97.1   98.2
KOL001     18.8  113.8  115.0
PUN001     14.5   91.6   92.3

largest difference vs the table dens_y: 0.0


In [24]:
# stock per product and branch at the end of the data: spread and the largest pair per branch
ps = pair['stock_end']
print("pairs:", len(ps), "| smallest:", int(ps.min()), "| mean:", round(ps.mean()), "| largest:", int(ps.max()))
print("largest vs mean:", round(ps.max() / ps.mean(), 2), "| smallest vs mean:", round(ps.min() / ps.mean(), 2))
print()
share = ps / ps.groupby('branch_id').transform('sum') * 100
print("three largest pairs and their share of the branch stock %:")
for (p, b), v in ps.sort_values(ascending=False).head(3).items():
    print(p, b, int(v), round(share[(p, b)], 2))
print()
print("largest single pair share of each branch stock %:")
print(share.groupby('branch_id').max().round(2).to_string())

pairs: 180 | smallest: 98642 | mean: 117696 | largest: 132967
largest vs mean: 1.13 | smallest vs mean: 0.84

three largest pairs and their share of the branch stock %:
P002 HYD001 132967 3.74
P012 DEL001 129402 3.6
P003 PUN001 129186 3.61

largest single pair share of each branch stock %:
branch_id
AHM001    3.60
CHN001    3.54
DEL001    3.60
HYD001    3.74
KOL001    3.55
PUN001    3.61


In [25]:
# earliest and latest dates in the data and how much falls in 2025
sold_date = so_h[so_h['order_status'] == 'Delivered'].set_index('so_id')['delivery_date']
bought_date = po_h[po_h['po_status'] == 'Received'].set_index('po_id')['received_date']
sl = so_l[so_l['so_id'].isin(sold_date.index)].copy()
pl = po_l[po_l['po_id'].isin(bought_date.index)].copy()
sl['d'] = pd.to_datetime(sl['so_id'].map(sold_date))
pl['d'] = pd.to_datetime(pl['po_id'].map(bought_date))
lg = pd.to_datetime(ledger['movement_date'])

print('delivered lines, earliest and latest delivery date:', sl['d'].min().date(), sl['d'].max().date())
print('received lines, earliest and latest received date:', pl['d'].min().date(), pl['d'].max().date())
print('stock ledger, earliest and latest movement date:', lg.min().date(), lg.max().date())
print()
print('delivered lines dated 2025:', (sl['d'].dt.year == 2025).sum(), '| units:', sl.loc[sl['d'].dt.year == 2025, 'quantity'].sum())
print('received lines dated 2025:', (pl['d'].dt.year == 2025).sum(), '| units:', pl.loc[pl['d'].dt.year == 2025, 'quantity'].sum())
print('ledger rows dated 2025:', (lg.dt.year == 2025).sum())
print('ledger rows dated after 2025:', (lg.dt.year > 2025).sum())

delivered lines, earliest and latest delivery date: 2019-01-03 2025-01-13
received lines, earliest and latest received date: 2019-01-11 2025-01-28
stock ledger, earliest and latest movement date: 2019-01-01 2025-01-28

delivered lines dated 2025: 340 | units: 3372
received lines dated 2025: 1246 | units: 194954
ledger rows dated 2025: 1466
ledger rows dated after 2025: 0


In [26]:
# 2025 ledger rows by type and how many 2025 order lines have a ledger row
lg25 = ledger[pd.to_datetime(ledger['movement_date']).dt.year == 2025]
print('ledger rows dated 2025 by type:')
print(lg25['movement_type'].value_counts())

rec25 = pl[pl['d'].dt.year == 2025]
del25 = sl[sl['d'].dt.year == 2025]
po_keys = pd.MultiIndex.from_frame(ledger.loc[ledger['reference_type'] == 'PO', ['reference_id', 'product_id']])
so_keys = pd.MultiIndex.from_frame(ledger.loc[ledger['reference_type'] == 'SO', ['reference_id', 'product_id']])
rec_keys = pd.MultiIndex.from_frame(rec25[['po_id', 'product_id']])
del_keys = pd.MultiIndex.from_frame(del25[['so_id', 'product_id']])

print()
print('received lines dated 2025:', len(rec25), '| with a ledger row:', rec_keys.isin(po_keys).sum(), '| without:', (~rec_keys.isin(po_keys)).sum())
print('delivered lines dated 2025:', len(del25), '| with a ledger row:', del_keys.isin(so_keys).sum(), '| without:', (~del_keys.isin(so_keys)).sum())

lg_in = lg25[lg25['reference_type'] == 'PO']
lg_out = lg25[lg25['reference_type'] == 'SO']
in_keys = pd.MultiIndex.from_frame(lg_in[['reference_id', 'product_id']])
out_keys = pd.MultiIndex.from_frame(lg_out[['reference_id', 'product_id']])
print('ledger IN rows dated 2025 with no received line dated 2025:', (~in_keys.isin(rec_keys)).sum())
print('ledger OUT rows dated 2025 with no delivered line dated 2025:', (~out_keys.isin(del_keys)).sum())

ledger rows dated 2025 by type:
movement_type
IN     1143
OUT     323
Name: count, dtype: int64

received lines dated 2025: 1246 | with a ledger row: 1151 | without: 95
delivered lines dated 2025: 340 | with a ledger row: 320 | without: 20
ledger IN rows dated 2025 with no received line dated 2025: 9
ledger OUT rows dated 2025 with no delivered line dated 2025: 6


In [27]:
# why the 2025 lines with a ledger row are more than the 2025 ledger rows that match them
for name, keys, all_keys, part in [('received', rec_keys, po_keys, lg_in), ('delivered', del_keys, so_keys, lg_out)]:
    part_keys = pd.MultiIndex.from_frame(part[['reference_id', 'product_id']])
    has_row = keys[keys.isin(all_keys)]
    distinct = has_row.drop_duplicates()
    print(name, 'lines with a ledger row:', len(has_row))
    print(name, 'distinct order and product pairs among them:', len(distinct))
    print(name, 'of those pairs, ledger row dated 2025:', distinct.isin(part_keys).sum(), '| ledger row dated another year:', (~distinct.isin(part_keys)).sum())
    print(name, 'ledger rows dated 2025 that match a 2025 line:', part_keys.isin(keys).sum())
    print()

received lines with a ledger row: 1151
received distinct order and product pairs among them: 1000
received of those pairs, ledger row dated 2025: 1000 | ledger row dated another year: 0
received ledger rows dated 2025 that match a 2025 line: 1134

delivered lines with a ledger row: 320
delivered distinct order and product pairs among them: 283
delivered of those pairs, ledger row dated 2025: 283 | ledger row dated another year: 0
delivered ledger rows dated 2025 that match a 2025 line: 317



In [28]:
# for each order and product pair in 2025, compare the number of order lines with the number of ledger rows
for name, lines, part, a in [('received', rec25, lg_in, 'po_id'), ('delivered', del25, lg_out, 'so_id')]:
    n_lines = lines.groupby([a, 'product_id']).size().rename('lines')
    n_rows = part.groupby(['reference_id', 'product_id']).size().rename('rows')
    n_rows.index.names = [a, 'product_id']
    t = pd.concat([n_lines, n_rows], axis=1).fillna(0)
    t = t[t['lines'] > 0]
    short = t[(t['rows'] > 0) & (t['rows'] < t['lines'])]
    print(name, 'pairs with 2025 lines:', len(t))
    print(name, 'pairs with no ledger row:', (t['rows'] == 0).sum(), '| lines in them:', t.loc[t['rows'] == 0, 'lines'].sum())
    print(name, 'pairs with fewer ledger rows than lines:', len(short), '| lines short:', (short['lines'] - short['rows']).sum())
    print(name, 'pairs with the same number:', (t['rows'] == t['lines']).sum(), '| with more ledger rows than lines:', (t['rows'] > t['lines']).sum())
    print()

received pairs with 2025 lines: 1095
received pairs with no ledger row: 95 | lines in them: 95.0
received pairs with fewer ledger rows than lines: 17 | lines short: 17.0
received pairs with the same number: 983 | with more ledger rows than lines: 0

delivered pairs with 2025 lines: 303
delivered pairs with no ledger row: 20 | lines in them: 20.0
delivered pairs with fewer ledger rows than lines: 3 | lines short: 3.0
delivered pairs with the same number: 280 | with more ledger rows than lines: 0



In [29]:
# what the 2025 ledger rows with no 2025 order line are in the order tables
for name, part, keys, lines, head, hid, status_col, date_col in [
    ('received', lg_in, rec_keys, po_l, po_h, 'po_id', 'po_status', 'received_date'),
    ('delivered', lg_out, del_keys, so_l, so_h, 'so_id', 'order_status', 'delivery_date')]:
    pk = pd.MultiIndex.from_frame(part[['reference_id', 'product_id']])
    odd = part[~pk.isin(keys)][['reference_id', 'product_id', 'movement_date']].rename(columns={'reference_id': hid}).copy()
    odd['in_order_lines'] = pd.MultiIndex.from_frame(odd[[hid, 'product_id']]).isin(pd.MultiIndex.from_frame(lines[[hid, 'product_id']]))
    st = head.set_index(hid)
    odd['status'] = odd[hid].map(st[status_col])
    odd[date_col] = odd[hid].map(st[date_col])
    print(name, 'ledger rows dated 2025 with no 2025 line:', len(odd))
    print(name, 'of them found in the order lines:', odd['in_order_lines'].sum())
    print(odd.to_string(index=False))
    print()

received ledger rows dated 2025 with no 2025 line: 9
received of them found in the order lines: 9
    po_id product_id movement_date  in_order_lines    status received_date
PO-448619       P003    2025-01-14            True Cancelled           NaN
PO-190792       P005    2025-01-22            True Cancelled           NaN
PO-968989       P018    2025-01-08            True Cancelled           NaN
PO-850081       P021    2025-01-01            True Cancelled           NaN
PO-217565       P022    2025-01-06            True Cancelled           NaN
PO-667371       P023    2025-01-17            True Cancelled           NaN
PO-482299       P027    2025-01-17            True Cancelled           NaN
PO-618520       P028    2025-01-02            True Cancelled           NaN
PO-926660       P029    2025-01-16            True Cancelled           NaN

delivered ledger rows dated 2025 with no 2025 line: 6
delivered of them found in the order lines: 6
    so_id product_id movement_date  in_order_lines 

In [31]:
# yearly units IN, units OUT, net adjustments and the stock they build
lg = ledger.copy()
lg['prev'] = lg.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
first = lg['prev'].isna()
opening = inv.set_index(['product_id', 'branch_id'])['opening_stock']
lg.loc[first, 'prev'] = opening.reindex(pd.MultiIndex.from_frame(lg.loc[first, ['product_id', 'branch_id']])).values
lg['change'] = lg['running_balance'] - lg['prev']
adj_rows = lg[lg['movement_type'] == 'ADJUSTMENT']
yr_adj = adj_rows.groupby(pd.to_datetime(adj_rows['movement_date']).dt.year)['change'].sum()
yr_in = pl.groupby(pl['d'].dt.year)['quantity'].sum()
yr_out = sl.groupby(sl['d'].dt.year)['quantity'].sum()
g = pd.concat([yr_in.rename('units_in'), yr_out.rename('units_out'), yr_adj.rename('net_adjustments')], axis=1).fillna(0)
g['change_in_stock'] = g['units_in'] - g['units_out'] + g['net_adjustments']
g['stock_at_year_end'] = inv['opening_stock'].sum() + g['change_in_stock'].cumsum()
g['units_in_per_1_out'] = (g['units_in'] / g['units_out']).round(1)
print(g.astype({'units_in': int, 'units_out': int, 'net_adjustments': int, 'change_in_stock': int, 'stock_at_year_end': int}).to_string())

      units_in  units_out  net_adjustments  change_in_stock  stock_at_year_end  units_in_per_1_out
2019   3556945     204586             -668          3351691            3384660                17.4
2020   3786822     211190              -45          3575587            6960247                17.9
2021   3776259     206935              372          3569696           10529943                18.2
2022   3682130     202716               48          3479462           14009405                18.2
2023   3752938     205578              622          3547982           17557387                18.3
2024   3637075     200562             -267          3436246           20993633                18.1
2025    194954       3372                0           191582           21185215                57.8


In [32]:
# three items rebuilt by hand from the raw tables
def stock_of(prod, br, last_day):
    last_day = pd.Timestamp(last_day)
    op = inv[(inv['product_id'] == prod) & (inv['branch_id'] == br)]['opening_stock'].iloc[0]
    po_ok = po_h[(po_h['po_status'] == 'Received') & (po_h['branch_id'] == br)]
    po_ok = po_ok[pd.to_datetime(po_ok['received_date']) <= last_day]
    got = po_l[(po_l['product_id'] == prod) & po_l['po_id'].isin(po_ok['po_id'])]['quantity'].sum()
    so_ok = so_h[(so_h['order_status'] == 'Delivered') & (so_h['branch_id'] == br)]
    so_ok = so_ok[pd.to_datetime(so_ok['delivery_date']) <= last_day]
    sent = so_l[(so_l['product_id'] == prod) & so_l['so_id'].isin(so_ok['so_id'])]['quantity'].sum()
    rows = ledger[(ledger['product_id'] == prod) & (ledger['branch_id'] == br)].copy()
    rows['change'] = rows['running_balance'].diff()
    rows.iloc[0, rows.columns.get_loc('change')] = rows['running_balance'].iloc[0] - op
    rows = rows[(rows['movement_type'] == 'ADJUSTMENT') & (pd.to_datetime(rows['movement_date']) <= last_day)]
    adj = rows['change'].sum()
    return int(op), int(got), int(sent), int(adj), int(op + got - sent + adj)

for prod, br, day, earlier in [('P002', 'HYD001', '2025-12-31', 132967), ('P029', 'HYD001', '2019-12-31', 13616)]:
    op, got, sent, adj, stock = stock_of(prod, br, day)
    print(prod, br, 'up to', day)
    print('  opening stock', op, '| units received', got, '| units delivered', sent, '| net adjustments', adj)
    print('  stock', stock, '| earlier figure', earlier, '| difference', stock - earlier)
    print()

tot = sum(stock_of(p, 'KOL001', '2025-12-31')[4] for p in products['product_id'])
size = int(branches.loc[branches['branch_id'] == 'KOL001', 'warehouse_capacity'].iloc[0].split()[0])
print('KOL001 stock', tot, '| earlier figure 3435814 | difference', tot - 3435814)
print('KOL001 warehouse size', size, 'sqft | units per sqft', round(tot / size, 1), '| earlier figure 115.0')

P002 HYD001 up to 2025-12-31
  opening stock 219 | units received 138227 | units delivered 5413 | net adjustments -66
  stock 132967 | earlier figure 132967 | difference 0

P029 HYD001 up to 2019-12-31
  opening stock 192 | units received 14461 | units delivered 1031 | net adjustments -6
  stock 13616 | earlier figure 13616 | difference 0

KOL001 stock 3435814 | earlier figure 3435814 | difference 0
KOL001 warehouse size 29880 sqft | units per sqft 115.0 | earlier figure 115.0
